## byLLM Script

This is a script based in byLLM to utilize Meaning Typed Programming in order to clean the fdic dataset. MTP is used to allow the api to interpret what method of cleaning should be done. This utilizes a subset of the 250-row dataset with a slice of 5 columns to account for token limits. The results of this cleaning is compared against our baseline manual-prompting in "DSCI644-Project/analysis". 

The model byLLM is referencing is Google Gemini 3 flash, as it is the latest flash model available for the free tier api. 

### Setup
- Import required libraries 
- Initialize api

In [1]:
import os
import pandas as pd
from byllm.lib import Model, by
import time
from dataclasses import dataclass, asdict

# This pulls the value from the system variable
apiKey = os.getenv('LLM_API_KEY')

if not apiKey:
    raise ValueError("Missing API Key! Please set the LLM_API_KEY environment variable.")

llm = Model(model_name="gemini/gemini-flash-latest" , api_key= apiKey)

### Import Dataset

Slice the first 5 columns for rate limitting. Current model should be run with 5 columns x 250 rows. With a paid model, this may be able to be increased

In [2]:
# Import Dataset

filePath = '../../data/fdic/fdic_250.csv'
df = pd.read_csv(filePath)

# Create df2 for a smaller sample in testing
df2 = df.iloc[:250, :5]

# Debug Commands
#df2.shape
#df.head()

### byLLM model integration
- Create dataclass to be used in byLLM method
- use method to clean entire df with a batching option for larger datasets
- call byLLM to clean values in a column as a sub-call in the previous method
- return the cleaned df

In [3]:
# Model Integration

@dataclass
class TextBatch:
    values: list[str]

@by(llm)
def clean_column_values(batch: TextBatch) -> TextBatch:
    """
    Clean this list of strings
    """
    ...

def clean_entire_dataframe(df: pd.DataFrame):
    cleaned_df = df.copy().astype(str)
    
    for col in df.columns:
        print(f"--- Cleaning Column: {col} ---")
        column_data = cleaned_df[col].tolist()
        new_column_values = []
        
        # Variable Batch Size (Set to Current Size 250)
        batch_size = 250
        for i in range(0, len(column_data), batch_size):
            chunk = column_data[i : i + batch_size]
            
            try:
                # Wrap in dataclass for byllm
                result = clean_column_values(TextBatch(values=chunk))
                new_column_values.extend(result.values)
                
                # Small sleep to respect the 15 RPM limit
                time.sleep(4) 
                
            except Exception as e:
                print(f"Error in {col} at index {i}: {e}")
                new_column_values.extend(chunk) # Keep original on failure
                time.sleep(20)
        
        # Update the column in our dataframe
        # Ensure the lengths match before assignment
        if len(new_column_values) == len(cleaned_df):
            cleaned_df[col] = new_column_values
            
    return cleaned_df

### Model Usage
- Run command to clean dataset
- track runtime in seconds

In [4]:
start_time = time.time() 

clean_data = clean_entire_dataframe(df2)

end_time = time.time()
duration = end_time - start_time
print(f"Execution time: {duration:.2f} seconds")

--- Cleaning Column: fdic_certificate_number ---

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

--- Cleaning Column: institution_name ---

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

--- Cleaning Column: state_name ---

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

--- Cleaning Column: fdic_id ---

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

--- Cleaning Column: docket ---

Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM

In [7]:
#Expected (250, 5)
clean_data.shape

(250, 5)

### Export Cleaned Dataset for testing

In [8]:
clean_data.to_csv('../../data/fdic/byllm_cleaned_fdic.csv', index=False)